# Text Retrieval Results

## Mappa semantica interattiva: Interface che permette di esplorare lo spazio degli embedding


In [4]:
import asyncio
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from qdrant_client import QdrantClient
from openai import OpenAI
import sys
import os

# Setup Path
sys.path.append(os.path.abspath(".."))
from config.settings import settings

# --- CONFIGURAZIONE ---
QDRANT_COLLECTION = settings.QDRANT_ENRICHED_COLLECTION_NAME
BACKGROUND_SAMPLES = 500  # Punti di contesto
TOP_K = 10                # Quanti risultati recuperare
USER_QUERY = "Dark bass with granular texture" # <--- LA TUA QUERY TESTUALE

async def visualize_semantic_search_with_table():
    print(f"--- ANALISI RETRIEVAL SEMANTICO: '{USER_QUERY}' ---")

    # Init Clients
    client_q = QdrantClient(host=settings.QDRANT_CONNECTION_HOST, port=settings.QDRANT_PORT)
    client_o = OpenAI(api_key=settings.MODEL_API_KEY)

    # 1. EMBEDDING DELLA QUERY
    print("Calcolo embedding query...")
    emb_res = client_o.embeddings.create(
        input=USER_QUERY,
        model=settings.MODEL_EMBEDDING_MODEL
    )
    query_vector = emb_res.data[0].embedding

    # 2. RETRIEVAL
    print("Esecuzione ricerca vettoriale...")
    search_res = client_q.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        using="text_vector",
        limit=TOP_K,
        with_payload=True,
        with_vectors=["text_vector"]
    )

    # 3. BACKGROUND
    print("Recupero contesto di fondo...")
    scroll_res, _ = client_q.scroll(
        collection_name=QDRANT_COLLECTION,
        limit=BACKGROUND_SAMPLES,
        with_payload=True,
        with_vectors=["text_vector"]
    )

    # --- PREPARAZIONE DATI ---
    vectors = []
    meta = []
    table_data = [] # Lista per la tabella

    # A. Query
    vectors.append(query_vector)
    meta.append({
        "name": "QUERY",
        "tooltip": f"<b>QUERY UTENTE</b><br>'{USER_QUERY}'",
        "score": 1.0,
        "type": "query"
    })

    # B. Risultati
    result_indices = []
    for rank, hit in enumerate(search_res.points, start=1):
        vec = hit.vector.get("text_vector") if isinstance(hit.vector, dict) else hit.vector
        if vec is None: continue

        vectors.append(vec)
        payload = hit.payload or {}

        # Estrazione Dati per Tabella e Grafico
        filename = payload.get("original_filename", "Unknown")
        orig_label = payload.get("original_label") or payload.get("original_filename", "N/A")
        ai_label = payload.get("ai_label", "N/A")

        # Estrazione Tags e Quality Score (Richiesti)
        tags = payload.get("ai_tags", [])
        tags_str = ", ".join(tags[:5]) if isinstance(tags, list) else str(tags)
        stored_quality = payload.get("clap_score", 0.0) # Label Quality

        # Tooltip HTML per il grafico
        tooltip_html = (
            f"<b>File:</b> {filename}<br>"
            f"<b>Score:</b> {hit.score:.3f}<br>"
            f"<br>"
            f"✅ <b>AI:</b> {ai_label}<br>"
            f"❌ <b>Orig:</b> <i>{orig_label}</i>"
        )

        meta.append({
            "name": filename,
            "tooltip": tooltip_html,
            "score": hit.score,
            "type": "result"
        })
        result_indices.append(len(vectors) - 1)

        # Aggiungiamo alla tabella (Stesse colonne dell'altro script)
        table_data.append({
            "Rank": rank,
            "Similarity Score": hit.score,
            #"Filename": filename, # Commentato come richiesto o puoi scommentarlo
            "AI Description": ai_label,
            "Original Label": orig_label,
            "Tags": tags_str,
            "Label Quality": stored_quality
        })

    # C. Background
    for hit in scroll_res:
        if hit.id in [r.id for r in search_res.points]: continue
        vec = hit.vector.get("text_vector") if isinstance(hit.vector, dict) else hit.vector
        if vec is None: continue

        vectors.append(vec)
        payload = hit.payload or {}
        bg_label = payload.get("ai_label") or payload.get("original_filename", "Unknown")

        meta.append({
            "name": "Background",
            "tooltip": f"Context: {bg_label}",
            "score": 0,
            "type": "noise"
        })

    # --- 4. VISUALIZZAZIONE TABELLA ---
    print("\n" + "="*80)
    print(f"RISULTATI TOP-{TOP_K} PER QUERY: '{USER_QUERY}'")
    print("="*80)

    df_results = pd.DataFrame(table_data)

    # Formattazione Stile Tabella
    pd.set_option('display.max_colwidth', None)

    # Colori: Blu per la similarità alla query, Verde per la qualità intrinseca della label
    display(df_results.style.background_gradient(subset=['Similarity Score'], cmap='Blues')
                            .background_gradient(subset=['Label Quality'], cmap='Greens')
                            .format({'Similarity Score': '{:.4f}', 'Label Quality': '{:.4f}'}))

    print("\nGenerazione Grafico 2D...")

    # --- 5. VISUALIZZAZIONE GRAFICO ---
    X = np.array(vectors)
    pca = PCA(n_components=min(50, len(X)))
    X_pca = pca.fit_transform(X)
    tsne = TSNE(n_components=2, perplexity=min(30, len(X)-1), random_state=42, init='pca', learning_rate='auto')
    X_2d = tsne.fit_transform(X_pca)

    fig = go.Figure()

    # Query -> Risultati (Linee)
    q_x, q_y = X_2d[0]
    for idx in result_indices:
        r_x, r_y = X_2d[idx]
        score = meta[idx]['score']
        fig.add_trace(go.Scatter(
            x=[q_x, r_x], y=[q_y, r_y],
            mode='lines',
            line=dict(color='rgba(255, 0, 0, 0.3)', width=score * 3),
            hoverinfo='none', showlegend=False
        ))

    # Background (Grigio)
    bg_indices = [i for i, m in enumerate(meta) if m['type'] == 'noise']
    fig.add_trace(go.Scatter(
        x=X_2d[bg_indices, 0], y=X_2d[bg_indices, 1],
        mode='markers',
        marker=dict(size=6, color='#bdc3c7', opacity=0.4),
        text=[meta[i]['tooltip'] for i in bg_indices],
        #name='Database Context', # Rimosso dalla legenda per pulizia
        hoverinfo='text',
        showlegend=False
    ))

    # Risultati (Colorati)
    res_x = [X_2d[i][0] for i in result_indices]
    res_y = [X_2d[i][1] for i in result_indices]
    res_scores = [meta[i]['score'] for i in result_indices]
    res_tips = [meta[i]['tooltip'] for i in result_indices]

    fig.add_trace(go.Scatter(
        x=res_x, y=res_y,
        mode='markers',
        marker=dict(
            size=15, color=res_scores, colorscale='Viridis',
            showscale=True, colorbar=dict(title="Similarity"),
            line=dict(color='white', width=2)
        ),
        text=res_tips, name='Results', hoverinfo='text'
    ))

    # Query (Stella)
    fig.add_trace(go.Scatter(
        x=[q_x], y=[q_y],
        mode='markers',
        marker=dict(size=25, color='red', symbol='star', line=dict(color='black', width=1)),
        name=f'Query: "{USER_QUERY}"', hoverinfo='name'
    ))

    fig.update_layout(
        title=f"<b>Semantic Retrieval Analysis</b><br>Query: <i>{USER_QUERY}</i>",
        width=1000, height=800, template="plotly_white",
        xaxis=dict(visible=False), yaxis=dict(visible=False),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    fig.show()

# Esecuzione
# asyncio.run(visualize_semantic_search_with_table())
await visualize_semantic_search_with_table()

--- ANALISI RETRIEVAL SEMANTICO: 'Dark bass with granular texture' ---
Calcolo embedding query...
Esecuzione ricerca vettoriale...
Recupero contesto di fondo...

RISULTATI TOP-10 PER QUERY: 'Dark bass with granular texture'


,Rank,Similarity Score,AI Description,Original Label,Tags,Label Quality
0,1,0.7151,"Digital Synth Bass with Coarse, Dirty Texture",Dirty Synth Bass - Coarse Hit,"electro, dynamic, hardstyle, hard, dark",0.4213
1,2,0.7067,"Analog Synth Bass with Hard, Dark, and Dry Texture",Phonk Bass Shot - Midnight Hit,"dynamic, hard, clean, analog, dark",0.2842
2,3,0.7005,"Analog Synth Bass Pad with Dark, Aggressive Texture",Dark Bass Pad Melody,"aggressive, dynamic, cold, angry, energetic",-0.0462
3,4,0.7001,"Analog Bass-Line with Dark, Hard, and Dry Texture",Dubstep Bass Line - Tough and Rough 2,"dynamic, excited, hard, clean, analog",0.5388
4,5,0.6995,"Sub-bass with Dark, Aggressive Tone and Dry Texture",Dark Sub Bass Lead,"aggressive, barking, clean, dark, dry",0.1080
5,6,0.6988,Digital Synth Bass with Dynamic and Dry Texture,Wonky Color Bass Shot,"dynamic, dry, energetic, clean",0.4115
6,7,0.6960,Sub-bass with smooth dynamic analog character and dry texture,Sub Bass Smooth,"dynamic, soft, hard, clean, analog",0.2445
7,8,0.6948,"Digital Synth Bass with Dark, Flowing Texture",Techno Synth Bass Line - Dark Mod Bass,"short, one shot, flowing, cold",0.2970
8,9,0.6936,Digital Synth Bass with Eerie and Dynamic Texture,Color Bass Line - Ascendance,"eerie, dynamic, gloomy, ambient, dark",0.3192
9,10,0.6936,Digital Synth Bass with Eerie and Dynamic Texture,Aggressive Color Bass Line,"eerie, dynamic, gloomy, ambient, dark",0.3206



Generazione Grafico 2D...


# Grafico a barre comparative: Per una query, visualizzazione dei top-k risultati con:

Barre di similarità (testo-audio e audio-audio)

Indicatore di confidence CLAP

Tags estrapolati dai vicini

In [7]:
import asyncio
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from qdrant_client import QdrantClient
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
import sys
import os

# Setup Path
sys.path.append(os.path.abspath("../.."))
from config.settings import settings
from rag.clap.model_handler import create_clap_model

# --- CONFIGURAZIONE ---
QDRANT_COLLECTION = settings.QDRANT_ENRICHED_COLLECTION_NAME
USER_QUERY = "Dark bass with granular texture" # <--- LA TUA QUERY
TOP_K = 8 # Numero di risultati da confrontare

async def plot_comparative_bars():
    print(f"--- ANALISI COMPARATIVA: '{USER_QUERY}' ---")

    # 1. Init Clienti e Modelli
    client_q = QdrantClient(host=settings.QDRANT_CONNECTION_HOST, port=settings.QDRANT_PORT)
    client_o = OpenAI(api_key=settings.MODEL_API_KEY)
    print("Caricamento CLAP (per validazione acustica)...")
    clap = create_clap_model()

    # 2. Embedding Query (OpenAI - Per il Retrieval)
    print("Calcolo embedding OpenAI...")
    emb_res = client_o.embeddings.create(
        input=USER_QUERY,
        model=settings.MODEL_EMBEDDING_MODEL
    )
    openai_query_vec = emb_res.data[0].embedding

    # 3. Embedding Query (CLAP - Per la validazione Audio)
    # Ci serve per vedere se l'audio trovato suona davvero come la query
    print("Calcolo embedding CLAP...")
    clap_query_vec = clap.get_text_embedding([USER_QUERY])[0]

    # 4. Esecuzione Ricerca
    print("Retrieval dei candidati...")
    search_res = client_q.query_points(
        collection_name=QDRANT_COLLECTION,
        query=openai_query_vec,
        using="text_vector", # Usiamo OpenAI per trovare i candidati
        limit=TOP_K,
        with_payload=True,
        with_vectors=["audio_vector"] # Ci serve l'audio per il confronto
    )

    # 5. Elaborazione Dati
    filenames = []
    retrieval_scores = [] # OpenAI Score
    acoustic_scores = []  # CLAP Score (Query vs Audio)
    confidences = []      # Internal CLAP Score (Label vs Audio)
    tags_list = []

    print("Analisi cross-modale dei risultati...")
    for hit in search_res.points:
        payload = hit.payload or {}
        fname = payload.get("original_filename", "Unknown")

        # Retrieval Score (OpenAI)
        retrieval_score = hit.score

        # Acoustic Match (CLAP: Query Text vs Result Audio)
        # Calcoliamo quanto questo audio "suona" come la query
        audio_vec = hit.vector.get("audio_vector") if isinstance(hit.vector, dict) else hit.vector
        if audio_vec is not None:
            # Cosine Similarity manuale
            # sim = dot(A, B) / (norm(A) * norm(B))
            # I vettori CLAP sono spesso normalizzati, ma ricalcoliamo per sicurezza
            sim = np.dot(clap_query_vec, audio_vec) / (np.linalg.norm(clap_query_vec) * np.linalg.norm(audio_vec))
            acoustic_score = float(sim)
        else:
            acoustic_score = 0.0

        # Tags (Prendiamo i primi 3 per non affollare)
        ai_tags = payload.get("ai_tags", [])
        tags_str = ", ".join(ai_tags[:3]) if ai_tags else "No tags"

        # Internal Confidence (Già calcolata in ingestion)
        confidence = payload.get("clap_score", 0.5)

        filenames.append(fname)
        retrieval_scores.append(retrieval_score)
        acoustic_scores.append(acoustic_score)
        confidences.append(confidence)
        tags_list.append(tags_str)

    # Ordiniamo per Retrieval Score (dal migliore al peggiore per il grafico)
    # Invertiamo perché i grafici a barre orizzontali partono dal basso
    filenames.reverse()
    retrieval_scores.reverse()
    acoustic_scores.reverse()
    confidences.reverse()
    tags_list.reverse()

    # --- VISUALIZZAZIONE ---
    fig = go.Figure()

    # Barra 1: Retrieval Score (Semantica)
    fig.add_trace(go.Bar(
        y=filenames,
        x=retrieval_scores,
        name='Semantic Match (OpenAI)',
        orientation='h',
        marker=dict(
            color='rgba(52, 152, 219, 0.8)', # Blu
            line=dict(color='rgba(52, 152, 219, 1.0)', width=1)
        ),
        text=[f"{s:.2f}" for s in retrieval_scores],
        textposition='auto',
        hovertemplate="<b>Semantic Match</b><br>Score: %{x:.3f}<br><i>Quanto la descrizione è simile alla query</i>"
    ))

    # Barra 2: Acoustic Match (Sonora)
    fig.add_trace(go.Bar(
        y=filenames,
        x=acoustic_scores,
        name='Acoustic Match (CLAP)',
        orientation='h',
        marker=dict(
            color='rgba(46, 204, 113, 0.8)', # Verde
            line=dict(color='rgba(46, 204, 113, 1.0)', width=1)
        ),
        text=[f"{s:.2f}" for s in acoustic_scores],
        textposition='auto',
        hovertemplate="<b>Acoustic Match</b><br>Score: %{x:.3f}<br><i>Quanto il suono corrisponde alla query</i>"
    ))

    # Aggiungiamo i TAGS come annotazioni a destra
    annotations = []
    for i, tags in enumerate(tags_list):
        annotations.append(dict(
            xref='paper', yref='y',
            x=1.02, y=i,
            xanchor='left',
            text=f"🏷 {tags}",
            font=dict(size=10, color="gray"),
            showarrow=False
        ))

    fig.update_layout(
        title=f"<b>Retrieval Analysis</b>: '{USER_QUERY}'<br>Confronto Semantico vs Acustico",
        barmode='group', # Barre affiancate
        xaxis=dict(title="Similarity Score (0-1)", range=[0, 1]),
        yaxis=dict(title="Retrieved Files"),
        annotations=annotations,
        width=1100,
        height=600 + (TOP_K * 20),
        margin=dict(r=200), # Spazio a destra per i tag
        template="plotly_white",
        legend=dict(orientation="h", y=1.02, x=0, xanchor="left")
    )

    fig.show()

# Esecuzione
# asyncio.run(plot_comparative_bars())
await plot_comparative_bars()

H:\music-ai\venvmusic\lib\site-packages\tqdm\auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html



--- ANALISI COMPARATIVA: 'Dark bass with granular texture' ---
Caricamento CLAP (per validazione acustica)...


H:\music-ai\venvmusic\lib\site-packages\torch\functional.py:534: UserWarning:

torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3596.)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Load our best checkpoint in the paper.
The checkpoint is already downloaded
Load Checkpoint...
Calcolo embedding OpenAI...
Calcolo embedding CLAP...
Retrieval dei candidati...
Analisi cross-modale dei risultati...
